In [2]:
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np


In [3]:
calls = pd.read_pickle("/ceph/MethDev/pbio/kay/data/calls_loose.pkl")

In [4]:
calls

,chr,start,end,X2,df,delta_max,hi_cluster,lo_cluster,delta_max_trim,top_cluster,...,df_loco,phi,pval,p_loco,qval,neighbor_support,dominance_blocked,reason,category,call_reason
85,1,77801,77900,283.205748,16,0.509852,16,0,0.258787,16,...,15.0,1.536786,0.000000e+00,1.479381e-03,0.000000e+00,0,False,no_neighbor_support,euc_gene,rescue_isolated
125,1,117201,117300,209.579299,16,0.469321,16,0,0.315957,16,...,15.0,1.536786,0.000000e+00,9.155010e-12,0.000000e+00,1,False,ok,euc_gene,main
129,1,121701,121800,121.351245,16,0.345870,4,16,0.321142,9,...,15.0,1.536786,2.556039e-10,7.882232e-08,1.349449e-08,1,False,ok,euc_gene,main
142,1,123101,123200,60.386475,16,0.280947,0,9,0.235865,9,...,15.0,1.536786,9.862091e-04,2.243899e-02,1.093957e-02,1,False,ok,euc_gene,main
294,1,243401,243500,69.454337,16,0.201690,4,9,0.186608,0,...,15.0,1.536786,1.295724e-04,1.617305e-02,1.967793e-03,2,False,weak_effect,euc_gene,main
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
304611,5,15242801,15242900,77.878127,16,0.302944,13,12,0.154973,12,...,15.0,1.544736,1.968215e-05,3.571681e-02,1.171208e-03,2,False,weak_effect,het_te,main
304613,5,15243001,15243100,86.855885,16,0.244768,8,9,0.176046,12,...,15.0,1.544736,2.232159e-06,2.235987e-03,1.974125e-04,2,False,weak_effect,het_te,main
304627,5,15259701,15259800,70.739222,16,0.413612,8,6,0.340276,6,...,15.0,1.544736,1.047749e-04,9.935832e-04,4.410364e-03,1,False,ok,het_te,main
304628,5,15259801,15259900,67.105317,16,0.465839,5,4,0.305060,4,...,15.0,1.544736,2.397276e-04,1.577884e-02,8.303919e-03,2,False,ok,het_te,main


In [5]:
import numpy as np
import pandas as pd

def calls_to_bed(
    calls: pd.DataFrame,
    bed_path: str,
    add_chr_prefix: bool = True,   # set False if your IGV genome uses "1..5" not "chr1..chr5"
):
    """
    Export calls to a 9-column BED with itemRgb coloring by call_reason.
    - Converts to 0-based, half-open as BED requires.
    - score = scaled -log10(qval), clipped to [0,1000].
    - itemRgb = color by call_reason.
    """
    df = calls.copy()

    # --- chrom naming ---
    chrom = df['chr'].astype(str)
    if add_chr_prefix:
        chrom = chrom.map(lambda x: f'chr{x}')

    # --- 0-based half-open coordinates for BED ---
    chromStart = df['start'].astype(int) - 1
    chromEnd   = df['end'].astype(int)

    # --- name field: short but informative ---
    name = (
        df['category'].astype(str)
        + ';Δ=' + df['delta_max_trim'].round(3).astype(str)
        + ';q=' + df['qval'].apply(lambda x: f'{x:.2e}')
        + ';' + df['call_reason'].astype(str)
    )

    # --- score: 0..1000 from -log10(q) ---
    score = -np.log10(np.clip(df['qval'].to_numpy(), 1e-300, 1.0))
    score = (score * 100).clip(0, 1000).astype(int)

    # --- color by call_reason (no fillna(tuple)!) ---
    color_map = {
        'main':                ( 33,150,243),  # blue
        'rescue_isolated':     (255,152,  0),  # orange
        'consensus_neighbors': ( 76,175, 80),  # green
        'consensus_override':  (156, 39,176),  # purple
        'none':                (120,120,120),  # gray
    }
    # Use dict.get to supply a tuple default, then stringify "R,G,B"
    rgb_tuples = df['call_reason'].map(lambda x: color_map.get(x, (120,120,120)))
    itemRgb = rgb_tuples.map(lambda t: f'{t[0]},{t[1]},{t[2]}')

    # --- required 9 BED fields ---
    bed = pd.DataFrame({
        'chrom': chrom,
        'chromStart': chromStart,
        'chromEnd': chromEnd,
        'name': name,
        'score': score,
        'strand': '.',           # no strand info; IGV requires a placeholder
        'thickStart': chromStart,  # set equal to block for simple features
        'thickEnd': chromEnd,
        'itemRgb': itemRgb,
    })

    bed.to_csv(bed_path, sep='\t', header=False, index=False)


In [6]:
calls_to_bed(calls, '/ceph/MethDev/pbio/kay/data/dmw_calls_loose.bed')